# 1. Instalando as Bibliotecas

In [3]:
!pip -q install \
langchain \
langchain-community \
langchain-groq \
langchain-text-splitters \
faiss-cpu \
sentence-transformers \
pypdf \
python-dotenv

# 2. Importando as bibliotecas

In [2]:
import os

from google.colab import userdata

# Documentos
from langchain_community.document_loaders import PyPDFDirectoryLoader

# Divisão de texto
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

# Banco Vetorial
from langchain_community.vectorstores import FAISS

# Modelo Groq
from langchain_groq import ChatGroq

# Prompt
from langchain_core.prompts import ChatPromptTemplate

# Chains
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

print("✅ Bibliotecas carregadas.")

/tmp/ipykernel_688/3443337812.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


✅ Bibliotecas carregadas.


# 3. Clonar o repositório do GitHub

In [4]:
!rm -rf PortfolioAI

!git clone https://github.com/elissouza2023/PortfolioAI.git

BASE_PATH = "/content/PortfolioAI"

KNOWLEDGE_PATH = f"{BASE_PATH}/knowledge_base"

VECTOR_PATH = f"{BASE_PATH}/vector_store"

os.makedirs(VECTOR_PATH, exist_ok=True)

print("✅ Repositório clonado.")

Cloning into 'PortfolioAI'...
remote: Enumerating objects: 268, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 268 (delta 10), reused 28 (delta 6), pack-reused 234 (from 1)
Receiving objects: 100% (268/268), 38.03 MiB | 31.84 MiB/s, done.
Resolving deltas: 100% (112/112), done.
✅ Repositório clonado.


# 4. Configurar API Key da Groq

In [5]:
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("✅ API Key carregada.")

✅ API Key carregada.


# 5. Carregando documentos da pasta knowledge_base

In [6]:
loader = PyPDFDirectoryLoader(KNOWLEDGE_PATH)

documents = loader.load()

print(f"\n📄 Total de documentos: {len(documents)}")

for doc in documents:
    print(doc.metadata["source"])


📄 Total de documentos: 46
/content/PortfolioAI/knowledge_base/formacao_academica.pdf
/content/PortfolioAI/knowledge_base/formacao_academica.pdf
/content/PortfolioAI/knowledge_base/formacao_academica.pdf
/content/PortfolioAI/knowledge_base/formacao_academica.pdf
/content/PortfolioAI/knowledge_base/competencias_tecnicas.pdf
/content/PortfolioAI/knowledge_base/competencias_tecnicas.pdf
/content/PortfolioAI/knowledge_base/competencias_tecnicas.pdf
/content/PortfolioAI/knowledge_base/competencias_tecnicas.pdf
/content/PortfolioAI/knowledge_base/competencias_tecnicas.pdf
/content/PortfolioAI/knowledge_base/competencias_comportamentais.pdf
/content/PortfolioAI/knowledge_base/competencias_comportamentais.pdf
/content/PortfolioAI/knowledge_base/competencias_comportamentais.pdf
/content/PortfolioAI/knowledge_base/competencias_comportamentais.pdf
/content/PortfolioAI/knowledge_base/competencias_comportamentais.pdf
/content/PortfolioAI/knowledge_base/curriculo.pdf
/content/PortfolioAI/knowledge_b

# 6. Dividindo os documentos em chunks

In [7]:
text_splitter = RecursiveCharacterTextSplitter(

    chunk_size=900,

    chunk_overlap=150,

    separators=[
        "\n\n",
        "\n",
        ".",
        "!",
        "?",
        " "
    ]
)

texts = text_splitter.split_documents(documents)

print(f"✅ Chunks criados: {len(texts)}")

✅ Chunks criados: 100


# 7. Criando Embeddings

In [8]:
embeddings = HuggingFaceEmbeddings(

    model_name="sentence-transformers/all-MiniLM-L6-v2"

)

/tmp/ipykernel_688/733390720.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# 8. Banco Vetorial

In [9]:
vector_store = FAISS.from_documents(

    texts,

    embeddings

)

vector_store.save_local(VECTOR_PATH)

print("✅ Banco vetorial criado.")

✅ Banco vetorial criado.


In [10]:
from google.colab import files
import os

# Compacta a pasta
!zip -r vector_store.zip /content/PortfolioAI/vector_store

# Faz o download
files.download("vector_store.zip")

  adding: content/PortfolioAI/vector_store/ (stored 0%)
  adding: content/PortfolioAI/vector_store/index.faiss (deflated 7%)
  adding: content/PortfolioAI/vector_store/index.pkl (deflated 71%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 9. Modelo Groq

In [11]:
MODEL_NAME = "qwen/qwen3.6-27b"

llm = ChatGroq(

    model=MODEL_NAME,

    temperature=0.3,

    max_tokens=1500

)

print("✅ Modelo carregado.")

✅ Modelo carregado.


## 10. Prompt do PortfolioAI

In [12]:
system_prompt = """
Você é o PortfolioAI.

Seu objetivo é responder perguntas sobre Elisângela de Souza.

REGRAS IMPORTANTES

• Utilize EXCLUSIVAMENTE as informações presentes no contexto.

• Nunca invente experiências.

• Nunca complete informações por conta própria.

• Caso não exista resposta no contexto, diga:

"Não encontrei essa informação na minha base de conhecimento.
Caso deseje mais detalhes, recomendo entrar em contato diretamente com Elisângela."

• Sempre escreva de forma profissional, porém em primeira pessoal como em uma entrevista, lembre-se de que vocÊ é o meu asistente pessoal. Por exemplo diga sou uma profissional...

• Manetenha racioninio fluído, linguagem clara, tom entusiastico. Mantenha o tom de conversa, como em uma entrevista de emprego.

• Sempre responda em português.

• Quando possível organize a resposta em tópicos.

Contexto:

{context}
"""

prompt = ChatPromptTemplate.from_messages(

    [

        ("system", system_prompt),

        ("human", "{input}")

    ]

)

print("✅ Prompt criado.")

✅ Prompt criado.


# 11. Chain RAG

In [13]:
question_answer_chain = create_stuff_documents_chain(

    llm,

    prompt

)

retriever = vector_store.as_retriever(

    search_kwargs={

        "k":6

    }

)

rag_chain = create_retrieval_chain(

    retriever,

    question_answer_chain

)

print("✅ RAG criado.")

✅ RAG criado.


# 12. Função para perguntas

In [14]:
def perguntar(pergunta):

    resposta = rag_chain.invoke(

        {

            "input": pergunta

        }

    )

    print("="*80)

    print("PERGUNTA")

    print(pergunta)

    print()

    print("RESPOSTA")

    print(resposta["answer"])

    print()

    print("FONTES UTILIZADAS")

    fontes = set()

    for doc in resposta["context"]:

        fontes.add(

            os.path.basename(

                doc.metadata["source"]

            )

        )

    for fonte in sorted(fontes):

        print("•", fonte)

    print("="*80)

# 13. Testes

In [15]:
perguntar("Quem é a Elisângela?")

PERGUNTA
Quem é a Elisângela?

RESPOSTA

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - User asks: "Quem é a Elisângela?" (Who is Elisângela?)
   - Language: Portuguese
   - Context provided: Contains a section specifically titled "Sobre Mim" (About Me) with the question "**Quem é Elisângela?**" and a detailed answer.

2.  **Extract Relevant Information from Context:**
   - Name: Elisângela de Souza
   - Location: Volta Redonda – RJ
   - Background: Combines solid operational and administrative experience in the industry with an active transition to Information Technology.
   - Education: Bachelor's in Administration, Postgraduate in Metallurgical Engineering, currently finishing Technology in Information Security.
   - Professional Goal: Work in IT, especially in Information Security, Technical Support/Cloud, Data Analysis, or projects combining administration and technology. Wants to combine practical experience with teamwork.
   - Philosophy/Approach: Solves pr

In [16]:
perguntar("Qual é o seu objetivo profissional ?")

PERGUNTA
Qual é o seu objetivo profissional ?

RESPOSTA

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** "Qual é o seu objetivo profissional ?" (What is your professional objective?)
   - **Language:** Portuguese
   - **Role:** PortfolioAI (acting as Elisângela de Souza's personal assistant, responding in first person as if in an interview)

2.  **Scan Context for Keywords:**
   - Keywords: "objetivo profissional", "objetivo", "meta", "busco"
   - Found in context: 
     "Objetivo Profissional
     Meu objetivo é atuar em projetos que permitam integrar Inteligência Artificial,
     desenvolvimento de software, dados e experiência do usuário para criar soluções
     inovadoras que gerem valor para pessoas e organizações.
     Busco contribuir com equipes colaborativas, compartilhando conhecimento, aprendendo
     continuamente e participando da construção de produtos digitais que aliem qualidade"

3.  **Extract Relevant Information:**
   - Goal: Act on

In [17]:
perguntar("Quais projetos ela desenvolveu?")

PERGUNTA
Quais projetos ela desenvolveu?

RESPOSTA

<think>
Here's a thinking process that leads to the suggested response:

1.  **Analyze the Request:**
    *   **User Question:** "Quais projetos ela desenvolveu?" (What projects did she develop?)
    *   **Persona:** PortfolioAI (Elisângela de Souza's personal assistant).
    *   **Tone:** Professional, first-person ("I"), enthusiastic, interview-like, fluid reasoning.
    *   **Constraints:** Use *only* provided context. Do not invent. Use bullet points if possible. Portuguese language.

2.  **Scan the Context for Keywords:**
    *   Keywords: "Projetos", "Cases", "Desenvolveu", "PortfolioAI", "Kaida", "Dashboard", "Flow State".

3.  **Extract Relevant Information:**
    *   *Source Section:* "Projetos em Destaque" (Featured Projects).
    *   *Project 1:* **PortfolioAI** (The main project).
        *   *Details:* Developed alone, Python, LangChain, Llama (via Groq), Hugging Face (embeddings), FAISS. 100% hands-on.
    *   *Project 2

In [18]:
perguntar("Fale sobre o projeto PortfolioAI.")

PERGUNTA
Fale sobre o projeto PortfolioAI.

RESPOSTA

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - User asks: "Fale sobre o projeto PortfolioAI." (Tell me about the PortfolioAI project.)
   - Language: Portuguese
   - Context provided: Contains information about Elisângela de Souza's projects, specifically mentioning "PortfolioAI – Base de Conhecimento do RAG" and details about it.

2.  **Scan Context for Keywords:**
   - Keywords: "PortfolioAI", "projeto", "desenvolvi", "tecnologias", "objetivo"
   - Found in context:
     - "Projeto: PortfolioAI – Base de Conhecimento do RAG"
     - "Objetivo do documento: Este documento reúne os principais projetos desenvolvidos por Elisângela de Souza..."
     - "O objetivo é permitir que o PortfolioAI responda perguntas como: Quais projetos você desenvolveu? Você possui experiência em Machine Learning? Já trabalhou com UX? Tem projetos de Segurança da Informação? Possui experiência com Python? Como costuma estruturar seus p

In [19]:
perguntar("Quais competências técnicas ela possui?")

PERGUNTA
Quais competências técnicas ela possui?

RESPOSTA

<think>
Here's a thinking process that leads to the suggested response:

1.  **Analyze the Request:**
    *   **User Question:** "Quais competências técnicas ela possui?" (What technical skills does she possess?)
    *   **Persona:** PortfolioAI (Elisângela de Souza's personal assistant).
    *   **Tone:** Professional, first-person ("I am a professional..."), enthusiastic, fluid, interview-style.
    *   **Language:** Portuguese.
    *   **Constraint:** Use *only* the provided context. Do not invent. If missing, use the specific fallback phrase. Organize in bullet points where possible.

2.  **Scan the Context for Keywords:**
    *   Keywords: "Competências Técnicas", "Tecnologias", "Desenvolvimento", "Ferramentas".
    *   *Found Section:* "COMPETÊNCIAS TÉCNICAS" (Technical Skills).
    *   *Found Section:* "Competências:" (Skills) near the end of the first block.
    *   *Found Section:* "Desenvolvimento de Software" (Softw

In [20]:
perguntar("Qual sua formação acadêmica?")

PERGUNTA
Qual sua formação acadêmica?

RESPOSTA

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** "Qual sua formação acadêmica?" (What is your academic background?)
   - **Language:** Portuguese
   - **Target:** Elisângela de Souza (as per the system prompt)

2.  **Scan Context for Keywords:**
   - Keywords: "formação acadêmica", "graduação", "cursos", "diplomas", "estudos"
   - Relevant sections in context:
     - "FORMAÇÃO ACADÊMICA"
     - "Este documento apresenta minha formação acadêmica formal..."
     - "Minha formação foi construída ao longo da carreira..."
     - "Acredito que a formação acadêmica representa muito mais do que a obtenção de diplomas."
     - "Cada curso realizado ampliou minha capacidade..."
     - "Graduação em Segurança da Informação" (mentioned under Evidências)
     - Mentions of continuous learning, certifications, complementary courses (referenced in "documento 04 – Formação Complementar")
     - No specific degrees, univ